# Bleed-through restoration

Set `IMAGE_PATH` in the first code cell and select **Run All**. Use a Python 3 kernel with NumPy, OpenCV and Matplotlib. If needed, install them once with `%pip install numpy opencv-python matplotlib`.

Choose a scan with bleed-through. The notebook restores every supplied scan and saves a black-and-white PNG beside it.

## Choose an image

The example uses a bundled H-DIBCO scan. Change the path to try another scan.

In [ ]:
IMAGE_PATH = "joeal_data/hdibco2014/images/H08.png"

## 1. Load the scan

OpenCV reads the scan as an 8-bit image. A relative path starts from the notebook kernel’s working folder. The original preview uses colour, and the algorithm uses grayscale.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

input_path = Path(IMAGE_PATH).expanduser()
if not input_path.is_absolute() and not input_path.is_file():
    input_path = Path.cwd().parent / input_path
input_path = input_path.resolve()
if not input_path.is_file():
    raise FileNotFoundError(f"Set IMAGE_PATH to an existing image file. Tried {input_path}")

original = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
if original is None:
    raise ValueError(f"OpenCV could not decode this image: {input_path}")
if min(original.shape[:2]) < 8:
    raise ValueError("The scan needs to be at least 8 pixels wide and tall.")

## Restoration methods

### Luminance

Convert BGR colour to one brightness value per pixel. Green gets the largest weight, followed by red and blue. The integer arithmetic rounds the weighted result back to an 8-bit pixel. A grayscale input passes through directly.

In [ ]:
def luminance(image):
    """Convert an OpenCV-order colour array to 8-bit grayscale."""
    if image.ndim == 2:
        return image
    blue, green, red = np.moveaxis(image.astype(np.int32), -1, 0)
    return ((3735 * blue + 19235 * green + 9798 * red + 16384) >> 15).astype(np.uint8)

### Histogram counts and sums

Count how often each brightness value appears. Build running totals for both pixel count and brightness so Otsu can calculate the size and mean of any intensity range by subtracting two totals.

In [ ]:
def _groups(image):
    counts = np.bincount(image.ravel(), minlength=256)
    levels = np.flatnonzero(counts)
    counts = counts[levels]
    weight = np.r_[0, np.cumsum(counts, dtype=np.float64)]
    moment = np.r_[0, np.cumsum(counts * levels, dtype=np.float64)]
    return levels, weight, moment

### Two-class Otsu

Try a cutoff after each occupied brightness level that leaves pixels on both sides. For each split, multiply the two group sizes by the squared difference between their average brightness. Return the cutoff with the highest score. The restoration pipeline uses pixels strictly below that cutoff as candidate ink. If every pixel has the same brightness, Otsu returns that brightness.

In [ ]:
def threshold_otsu(image):
    """Pick the split that maximises the weighted separation of two groups."""
    levels, weight, moment = _groups(image)
    if len(levels) == 1:
        return int(levels[0])

    left = weight[1:-1]
    right = weight[-1] - left
    left_mean = moment[1:-1] / left
    right_mean = (moment[-1] - moment[1:-1]) / right
    separation = left * right * (left_mean - right_mean) ** 2
    return int(levels[np.argmax(separation)])

### Small-component removal

Group candidate ink pixels that touch up, down, left or right. Count the pixels in each group and discard groups with 30 pixels or fewer at the default setting. The result is a mask of the larger candidate marks.

In [ ]:
def remove_specks(mask, max_size):
    """Keep 4-connected foreground components larger than max_size."""
    _, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=4
    )
    keep = stats[:, cv2.CC_STAT_AREA] > max_size
    keep[0] = False
    return keep[labels]

### Restoration settings

These defaults set the size of the background filters, the speck cutoff, the Canny thresholds and the minimum edge overlap.

In [ ]:
PARAMS = dict(
    close_size=31,       # step 1: closing kernel, larger than any stroke width
    median_size=21,      # step 1: smooths the background estimate
    min_speck=30,        # step 2: components of this many pixels or fewer are removed
    canny=(60, 150),     # step 3: Canny thresholds on the original image
    min_edge_ratio=0.08, # step 3: edge pixels / component size needed to count as real ink
)

### Background normalisation

Estimate what the paper would look like with the writing filled in. Morphological closing with a 31 × 31 ellipse fills narrow dark marks using nearby brighter pixels. A 21-pixel median filter smooths that estimate. The code divides each original pixel by its estimated background and multiplies by 255. Paper close to its local background becomes nearly white, while ink stays darker in proportion to the paper around it.

In [ ]:
def bg_normalise(gray, close_size, median_size):
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size))
    background = cv2.medianBlur(cv2.morphologyEx(gray, cv2.MORPH_CLOSE, se), median_size)
    norm = np.clip(gray.astype(float) / np.maximum(background, 1) * 255, 0, 255).astype(np.uint8)
    return norm, background

### Canny edge gate

Lightly blur the original grayscale scan and use Canny with thresholds of 60 and 150 to find strong brightness changes. Label groups of candidate ink, including diagonal neighbours. For each group, divide the number of its pixels that overlap a Canny edge by its total pixel count. Keep the entire group when that fraction is greater than 8%. This favours strokes with clear edges and can also discard genuine faint writing.

In [ ]:
def edge_gate(binary, gray, canny, min_edge_ratio):
    edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), *canny) > 0
    nl, lab = cv2.connectedComponents(binary.astype(np.uint8), connectivity=8)
    edge_cnt = np.bincount(lab[edges], minlength=nl)
    size = np.bincount(lab.ravel(), minlength=nl)
    keep = edge_cnt / np.maximum(size, 1) > min_edge_ratio
    keep[0] = False                                   # label 0 is background
    return keep[lab], edges

### Put the restoration steps together

Estimate and correct the paper background, apply Otsu to identify candidate ink, remove small specks, then check the remaining groups against the original scan’s edges. The function returns the final ink mask and intermediate images for inspection.

In [ ]:
def binarize(gray, p=PARAMS):
    norm, background = bg_normalise(gray, p['close_size'], p['median_size'])          # step 1
    t = threshold_otsu(norm)
    thresh = remove_specks(norm < t, p['min_speck'])                  # step 2
    final, edges = edge_gate(thresh, gray, p['canny'], p['min_edge_ratio'])           # step 3
    return final, dict(background=background, normalised=norm, otsu_t=t,
                       thresholded=thresh, edges=edges, removed_by_gate=thresh & ~final)

## Save and compare

Convert the final mask into black ink on white paper, save a PNG beside the input, and display both images. Running the notebook again for the same input replaces its previous restored PNG.

In [ ]:
mask, restoration_steps = binarize(luminance(original))
output = np.where(mask, 0, 255).astype(np.uint8)
output_path = input_path.with_name(f"{input_path.name}.bleedthrough-restored.png")
if not cv2.imwrite(str(output_path), output):
    raise OSError(f"Could not save the restored image to {output_path}")
print(f"Saved restored image at {output_path}")

figure, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original scan")
axes[1].imshow(output, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Restored scan")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()
plt.close(figure)